## Combining ACS 5 Year Metrics and Property Data

This notebook will require you to have ingested block groups (notebooks, 02_ingest, tiles, ingest_tiles), any parcel layer (notebooks, 02_ingest, parcels, ingest_parcels), and the census data (notebooks, 02_ingest, population, ingest_population). 

Here are the census metrics available to you: 

- b01001   # Sex by Age
- b19013   # Median Household Income
- b02001   # Race
- b25077   # Median Property Value (Dollars)
- b17001   # Poverty Status in the Past 12 Months
- b15003   # Educational attainment for over 25yo
- b23025   # Employment status (Labor force, unemployment) 
- b08303   # Travel time to work
- b25002   # Occupancy Status 
- b25064   # Median Gross Rent
- b25003   # Tenure (Owner vs Renter)
- b25091   # Mortgage Status by Selected Monthly Owner Costs
- b25070   # Gross Rent as % of Household Income (Rent Burden)
- b03002   # Hispanic or Latino Origin by Race
- c16002   # Household Language by English Proficiency
- b11004   # Family Type by Presence of Own Children
- b01003   # Total Population
- b22010   # Receipt of Food Stamps/SNAP

### First, we import our ACS data for desired metrics. 

In [ ]:
# Get ACS data for selected metrics
from openplaces.api import get_dataset
import pandas as pd

partitions = [
    'b19013',  # Median Household Income
    'b02001',  # Race
    'b25077',  # Median Property Value (Dollars)
    'b17001',  # Poverty Status
    'b15003',  # Educational attainment
    'b23025',  # Employment status
    'b08303',  # Travel time to work
    'b25064',  # Median Gross Rent
    'b25070',  # Rent Burden
    'b03002',  # Hispanic or Latino Origin by Race
]

dfs = []

for pid in partitions:
    df = get_dataset(recipe='US_population-acs-2024', partition_id=pid)

    # ensure index is set
    if 'census_geo_id' in df.columns:
        df = df.set_index('census_geo_id')

    dfs.append(df)

acs = pd.concat(dfs, axis=1, join='outer')

In [ ]:
acs.head(1)

### Next, we import our block group data, and merge on census_geo_id to the ACS metrics

In [ ]:
# Get block groups
from openplaces.api import get_entities

bg = get_entities(recipe='US_tile-census-2025_blockgroup', geom=True)
ma_bg = bg[bg['admin2_id'] == 'US-MA']

In [ ]:
import numpy as np

ma_census = ma_bg.merge(acs, on='census_geo_id', how='left')

# exclude ID + geometry columns from numeric operations
exclude = [
    'census_geo_id',
    'geometry',
    'admin3_id_admin1',
    'admin3_id',
    'admin2_id',
    'geometry',
]
work_cols = ma_census.columns.difference(exclude)

# ensure numeric only where appropriate
ma_census[work_cols] = ma_census[work_cols].apply(pd.to_numeric, errors='coerce')

# remove ACS negative codes only on numeric columns
ma_census[work_cols] = ma_census[work_cols].mask(ma_census[work_cols] < 0, np.nan)

# geometry cleaning
ma_census = ma_census[ma_census.geometry.notna()]
ma_census = ma_census[ma_census.geometry.is_valid]

### Check columns that have high amounts of null values

In [ ]:
null_counts = ma_census.isna().sum().sort_values(ascending=False)
null_counts

In [ ]:
ma_census.columns

### Color plot of the 3 metrics we pulled in using matplotlib and geopandas. 

In [ ]:
ma_census['percent_rent_burdened'] = (
    ma_census['rent_burden_30_to_34']
    + ma_census['rent_burden_35_to_39']
    + ma_census['rent_burden_40_to_49']
    + ma_census['rent_burden_ge_50']
) / ma_census['renter_households']

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd

fig, axes = plt.subplots(1, 3, figsize=(15, 3))

ma_census.plot(column='percent_rent_burdened', ax=axes[0], legend=True, cmap='Reds')
axes[0].set_title('Percent of Rent Burdened Households')
axes[0].axis('off')

ma_census.plot(column='med_household_inc', ax=axes[1], legend=True, cmap='Greens')
axes[1].set_title('Median Gross Rent')
axes[1].axis('off')

ma_census.plot(column='median_home_val', ax=axes[2], legend=True, cmap='viridis_r')
axes[2].set_title('Median Property Value')
axes[2].axis('off')

plt.tight_layout()
plt.show()

### Next, we can pull in our Massachusetts properties. 

In [ ]:
from openplaces.api import get_entities

prop = get_entities(
    recipe='US-MA_parcel-massgis-2025', admin_id='US-MA-NO', layer='property'
)
parcel = get_entities(
    recipe='US-MA_parcel-massgis-2025', admin_id='US-MA-NO', geom=True
)

In [ ]:
parcel_geom = parcel[['parcel_id_admin2', 'geometry']]

# merge onto prop
prop_merged = prop.merge(parcel_geom, on='parcel_id_admin2', how='left')

prop_merged = gpd.GeoDataFrame(prop_merged, geometry='geometry')

### We can use a spatial join to connect our property attributes to census data.

In [ ]:
import geopandas as gpd

prop_census = gpd.sjoin(prop_merged, ma_census, how='left', predicate='intersects')

In [ ]:
prop_census['last_sale_date'] = pd.to_datetime(
    prop_census['last_sale_date'], errors='coerce'
)

In [ ]:
single_fam = prop_census[prop_census['usecode'].isin(['101', '1010'])]

In [ ]:
single_fam[
    single_fam['last_sale_price'].notna() & (single_fam['last_sale_price'] > 0)
].copy()

In [ ]:
import matplotlib.pyplot as plt

ax = parcel.plot(color='lightgrey', figsize=(10, 10), alpha=0.5)

high_value = single_fam[single_fam['median_home_val'] > 800_000]

recent = single_fam[single_fam['last_sale_date'] > '2023-01-01']

affordable = recent[recent['last_sale_price'] < 500_000]

mid_market = recent[
    (recent['last_sale_price'] >= 500_000) & (recent['last_sale_price'] <= 800_000)
]

unaffordable = recent[recent['last_sale_price'] > 800_000]

high_value.plot(ax=ax, color='blue', alpha=0.25)

affordable.plot(ax=ax, color='green', markersize=5)
mid_market.plot(ax=ax, color='orange', markersize=5)
unaffordable.plot(ax=ax, color='red', markersize=5)

plt.title(
    'Single-Family Housing Price Composition in High-Value Neighborhoods (Since 2023)'
)
plt.axis('off')

summary_text = (
    f'Affordable Sales (<$500k): {len(affordable):,}\n'
    f'Mid-market Sales ($500k–$800k): {len(mid_market):,}\n'
    f'Unaffordable Sales (>$800k): {len(unaffordable):,}'
)

plt.figtext(0.5, 0.01, summary_text, ha='center', fontsize=10)
plt.subplots_adjust(bottom=0.15)

plt.show()